# Create MDA Awards (Muscular Dystrophy Association)

Creates Muscular Dystrophy Association research-grant awards from MDA's own
public "Grants at a Glance" database at
https://www.mda.org/science/grants-at-a-glance (Drupal view, ~25 grants per
page, 31 pages). Each row carries title, grantee, funded dates, grant type,
disease(s), lay abstract, award total (USD), institution, country and, for
grants from 2016 on, a Crossref grant DOI (prefix 10.55762).
**763 grants, 2010-2026, USD 125.4M** (local run 2026-09-30). The view lists
4 grants twice under two node ids (same title, PI, dates, amount); the script
keeps one copy (the one carrying the Crossref award number, else the lowest node id).

**Legacy rows (2010-2019, ~457 rows):** MDA's older records are news-style:
the title is a placeholder ("Grant - Summer 2015 - CMT - Daniel Summers,
Ph.D."), dates/amount/institution fields are blank. The script:
- takes the PI from the grantee string (drops the disease prefix),
- sets `start_date` from the "Funding ... began Aug. 1, 2015" sentence, else the
  first month of the round (Winter=Jan, Spring=Apr, Summer=Jul),
- takes `amount` from the record's fixed sentence "was awarded an MDA research
  grant totaling $300,000 over three years" (`amount_source = 'description'`;
  `award_total` for the structured field),
- ships a composed `display_name` "MDA Research Grant, Summer 2015 (CMT: Daniel
  Summers)" (`title_is_composed = 'true'`); the source title is kept in
  `raw_title`.

**`funder_award_id` (section 2.1.1):** MDA registers grant DOIs with Crossref;
the deposited award number is the `<award>` segment of the DOI
(`10.55762/mda.1446047.pc.gr.235453` -> `1446047`). 262 of these are already in
`openalex_awards_raw` as `crossref_work` stubs (priority 1) keyed on that bare
number. The script uses it for 235 grants (DOI on the page, or an unambiguous
exact-title match to MDA's Crossref deposits where the page omits the DOI), so
those rows upgrade 235 of the 262 stubs in CreateAwards dedup (verified). Citing works
also write `MDA603238` / bare 6-digit numbers for older grants, but MDA's site
prints no grant number for grants without a DOI, so those 528 rows ship
`MDA-NID-<drupal node id>` (MDA's own stable record id; stable across runs).

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320308306`
- display_name: Muscular Dystrophy Association
- ror_id / doi: from `openalex.funders.funders` (ror 01frxsf98, doi 10.13039/100005202)

**Prerequisites:** run `scripts/local/mda_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/mda/mda_projects.parquet` (section 1.4 shrink guard).

**Priority:** `498` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mda_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/mda/mda_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_grants, COUNT(DISTINCT funder_award_id) as distinct_award_ids FROM openalex.awards.mda_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.mda_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.mda_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320308306 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320308306;

## Step 2: Create MDA Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mda_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320308306  -- Muscular Dystrophy Association
),

g AS (
    SELECT
        *,
        -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
        CASE WHEN NULLIF(TRIM(lead_family_name), '') IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(lead_given_name), ''),
            'family_name', NULLIF(TRIM(lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(institution), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS lead
    FROM openalex.awards.mda_raw
    WHERE funder_award_id IS NOT NULL AND TRIM(funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.display_title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'USD' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    NULLIF(TRIM(g.grant_type), '') as funder_scheme,
    'mda_grants_at_a_glance' as provenance,
    TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
    TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
    YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
    YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
    g.lead as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN g.lead IS NOT NULL THEN array(g.lead) END as investigators,
    g.landing_page_url as landing_page_url,
    CASE WHEN NULLIF(TRIM(g.grant_doi), '') IS NOT NULL THEN CONCAT('https://doi.org/', LOWER(TRIM(g.grant_doi))) END as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Shape check (section 2.1.1): every funder_award_id is either MDA's Crossref
-- award number (bare digits) or the MDA-NID-<node> fallback. Fails the run otherwise.
SELECT assert_true(
    COUNT_IF(NOT (funder_award_id RLIKE '^[0-9]{5,8}$' OR funder_award_id RLIKE '^MDA-NID-[0-9]+$')) = 0
    AND COUNT(*) = COUNT(DISTINCT id),
    'mda_awards: unexpected funder_award_id shape or duplicate ids'
) AS shape_ok
FROM openalex.awards.mda_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'mda_grants_at_a_glance' AND priority = 498;

-- Insert into openalex_awards_raw with priority.
-- Priority 498: direct-from-funder ingest of MDA's own grants database.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500), so 498 outranks
-- the crossref_work grant-DOI stubs (priority 1) and acknowledgement shells
-- (priority 0) that share these award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    498 as priority
FROM openalex.awards.mda_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_mda_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.mda_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, doi
FROM openalex.awards.mda_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_usd
FROM openalex.awards.mda_awards
GROUP BY funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_usd
FROM openalex.awards.mda_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.mda_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.mda_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(doi) as has_doi,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.mda_awards;

In [ ]:
%sql
-- Overlap with the existing Crossref grant-DOI stubs: expect ~235 of the 262
-- crossref_work MDA rows to share an id with this ingest (these upgrade).
SELECT
    COUNT(*) as crossref_mda_rows,
    COUNT(t.id) as upgraded_by_mda_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.mda_awards t ON t.id = c.id
WHERE c.funder_id = 4320308306 AND c.provenance = 'crossref_work';

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'mda_grants_at_a_glance'
GROUP BY provenance, priority;